# 1. Environment Setup & Data Ingestion
**Purpose:** Connects the notebook session to Google Drive and extracts the compressed Mendeley dataset directly onto Colab's high-speed local disk (`/content/`).

* **Mounting:** Grants access to the raw archive stored in Drive.
* **Extraction:** Moving and unzipping locally prevents runtime disconnects and Drive I/O bottlenecks during model training.

In [ ]:
# SUMMARY: Mounts Google Drive and extracts the raw Mendeley dataset zip file
# directly to the local Colab runtime for faster processing.

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ZIP_PATH = "/content/drive/MyDrive/Files/Thesis Documents/mendeley_raw_data.zip"
LOCAL_SOURCE_DIR = "/content/mendeley_raw_data/Raw Data"

!cp "{DRIVE_ZIP_PATH}" /content/
!unzip -q /content/mendeley_raw_data.zip -d /content/mendeley_raw_data

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
replace /content/mendeley_raw_data/Laptop Components Image Dataset to Classify Different Components/Readme.pdf? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


# 2. Data Cleaning & Stratified Partitioning (ETL)
**Purpose:** Scans the raw component directories, verifies image integrity, and partitions the dataset.

* **Target Scope:** Limits ingestion strictly to the 5 selected laptop components (Motherboard, Battery, RAM, WiFi Card, CPU Fan).
* **Filtering:** Verifies that all files are valid and readable images (bypassing blur drops to preserve all 140 images per class).
* **Splitting:** Executes a stratified split to ensure an exact 80% Train, 10% Validation, and 10% Test ratio across every component class.

In [ ]:
# SUMMARY: Executes the Extract, Transform, Load (ETL) pipeline. Applies an 80/10/10
# stratified split and organizes raw images into standardized training directories.

import os
import cv2
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

DIR_TO_CLASS = {
    "5. Motherboard": "Motherboard",
    "1. Battery": "Battery",
    "8. RAM": "RAM",
    "11. WifiCard": "WiFi Card",
    "20. CPUFan": "CPU Fan"
}
TARGET_CLASSES = list(DIR_TO_CLASS.values())
SOURCE_DIR = "/content/mendeley_raw_data/Laptop Components Image Dataset to Classify Different Components/Raw Data/Raw Data"
BASE_OUTPUT_DIR = "/content/structured_ewaste_pipeline"
SPLITS = ["train", "val", "test"]
BLUR_THRESHOLD = 100.0
MIN_CLASS_SIZE = 120
VAL_TEST_SIZE = 0.20
RANDOM_STATE = 42

def calculate_laplacian_variance(image_path):
    image = cv2.imread(image_path)
    if image is None:
        return 0.0, False
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var(), True

def build_directory_structure():
    for split in SPLITS:
        for dir_name, component in DIR_TO_CLASS.items():
            os.makedirs(os.path.join(BASE_OUTPUT_DIR, split, component), exist_ok=True)

def process_and_split_dataset():
    dataset_records = []
    rejected_blurry = 0
    rejected_unreadable = 0

    for dir_name, component in DIR_TO_CLASS.items():
        component_dir = os.path.join(SOURCE_DIR, dir_name)
        if not os.path.exists(component_dir):
            continue

        filenames = [f for f in os.listdir(component_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        for filename in tqdm(filenames, desc=component):
            file_path = os.path.join(component_dir, filename)
            blur_variance, read_ok = calculate_laplacian_variance(file_path)

            if not read_ok:
                rejected_unreadable += 1
                continue

            if blur_variance >= BLUR_THRESHOLD:
                dataset_records.append({"file_path": file_path, "label": component})
            else:
                rejected_blurry += 1

    df = pd.DataFrame(dataset_records)
    if df.empty:
        return None

    class_counts = df['label'].value_counts()
    for component, count in class_counts.items():
        if count < MIN_CLASS_SIZE:
            raise ValueError(f"Class '{component}' has only {count} valid images, falling below the required floor of {MIN_CLASS_SIZE}.")

    train_df, temp_df = train_test_split(df, test_size=VAL_TEST_SIZE, stratify=df['label'], random_state=RANDOM_STATE)
    val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=RANDOM_STATE)

    partitions = [("train", train_df), ("val", val_df), ("test", test_df)]

    for split_name, partition_df in partitions:
        for _, row in tqdm(partition_df.iterrows(), total=len(partition_df), desc=split_name):
            src = row['file_path']
            dest = os.path.join(BASE_OUTPUT_DIR, split_name, row['label'], os.path.basename(src))
            shutil.copy2(src, dest)

    return df

build_directory_structure()
df = process_and_split_dataset()

5. Motherboard:   0%|          | 0/140 [00:00<?, ?it/s]

1. Battery:   0%|          | 0/140 [00:00<?, ?it/s]

8. RAM:   0%|          | 0/140 [00:00<?, ?it/s]

11. WifiCard:   0%|          | 0/140 [00:00<?, ?it/s]

20. CPUFan:   0%|          | 0/140 [00:00<?, ?it/s]

train:   0%|          | 0/560 [00:00<?, ?it/s]

val:   0%|          | 0/70 [00:00<?, ?it/s]

test:   0%|          | 0/70 [00:00<?, ?it/s]

# 3. TensorFlow Dataset Pipeline & Augmentation
**Purpose:** Converts raw directory paths into batched, memory-cached TensorFlow `tf.data` streams and sets up real-time augmentation.

* **Input Dimensions:** Resizes all samples to uniform $224 \times 224 \times 3$ tensors.
* **Prefetching:** Uses `AUTOTUNE` prefetching to ensure the GPU never sits idle waiting for batches.
* **Spatial Augmentation:** Applies randomized flips, rotations, translations, zooms, and contrast shifts during training to prevent overfitting.

In [ ]:
# SUMMARY: Loads images into batched TensorFlow Datasets, configures GPU caching,
# and builds the dynamic spatial data augmentation layer for the training pipeline.

import tensorflow as tf
from tensorflow.keras import layers

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16

train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_OUTPUT_DIR, "train"),
    labels="inferred",
    label_mode="categorical",
    class_names=TARGET_CLASSES,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=True,
    seed=RANDOM_STATE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_OUTPUT_DIR, "val"),
    labels="inferred",
    label_mode="categorical",
    class_names=TARGET_CLASSES,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_OUTPUT_DIR, "test"),
    labels="inferred",
    label_mode="categorical",
    class_names=TARGET_CLASSES,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=False
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(factor=(-30.0 / 360.0, 30.0 / 360.0)),
    layers.RandomZoom(height_factor=(-0.15, 0.15)),
    layers.RandomTranslation(height_factor=(-0.10, 0.10), width_factor=(-0.10, 0.10)),
    layers.RandomBrightness(factor=(-0.25, 0.25)),
    layers.RandomContrast(factor=0.20)
], name="e_waste_augmentation")

Found 560 files belonging to 5 classes.
Found 70 files belonging to 5 classes.
Found 70 files belonging to 5 classes.


# 4. Loss Function, Checkpoints & Model Builders
**Purpose:** Defines shared training infrastructure and model definitions for both DenseNet-121 and MobileNet-V2.

* **Custom Focal Loss:** Implements Categorical Focal Loss ($\gamma = 2.0, \alpha = 0.25$) to handle class complexity and hard-to-classify examples.
* **Drive Callbacks:** Implements `ModelCheckpoint` to save peak weights directly back to Google Drive and `EarlyStopping` to halt training if validation loss stalls.
* **Classification Head:** Attaches an identical top (Global Average Pooling, 256-ReLU Dense layer, 50% Dropout, and 5-class Softmax) to both backbone extractors.

In [ ]:
# SUMMARY: Defines the Custom Focal Loss function, training callbacks (early stopping
# and Drive checkpointing), and the base architectures for DenseNet-121 and MobileNet-V2.

import os
import tensorflow.keras.backend as K
from tensorflow.keras.applications import DenseNet121, MobileNetV2
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess

class CategoricalFocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.25, name="categorical_focal_loss"):
        super(CategoricalFocalLoss, self).__init__(name=name)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        epsilon = K.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)
        cross_entropy = -y_true * tf.math.log(y_pred)
        focal_weight = self.alpha * tf.math.pow(1.0 - y_pred, self.gamma)
        loss = focal_weight * cross_entropy
        return tf.reduce_mean(tf.reduce_sum(loss, axis=-1))

CHECKPOINT_DIR = "/content/drive/MyDrive/Thesis Documents/Model_Checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def get_training_callbacks(model_name):
    checkpoint = tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(CHECKPOINT_DIR, f"{model_name}_best.keras"),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    )
    best_val_loss = get_best_val_loss_so_far(model_name)
    if best_val_loss is not None:
        checkpoint.best = best_val_loss

    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
    csv_logger = tf.keras.callbacks.CSVLogger(
        filename=os.path.join(CHECKPOINT_DIR, f"{model_name}_log.csv"),
        append=True
    )
    return [checkpoint, early_stopping, csv_logger]

import pandas as pd
def get_best_val_loss_so_far(model_name):
    log_path = os.path.join(CHECKPOINT_DIR, f"{model_name}_log.csv")
    if os.path.exists(log_path):
        try:
            df = pd.read_csv(log_path)
            if not df.empty and 'val_loss' in df.columns:
                return df['val_loss'].min()
        except Exception:
            pass
    return None

def get_last_completed_epoch(model_name):
    log_path = os.path.join(CHECKPOINT_DIR, f"{model_name}_log.csv")
    if os.path.exists(log_path):
        try:
            df = pd.read_csv(log_path)
            if not df.empty:
                return int(df['epoch'].iloc[-1]) + 1
        except Exception:
            pass
    return 0

def build_densenet_model(name="DenseNet-121", apply_augmentation=True):
    inputs = layers.Input(shape=(224, 224, 3))
    x = data_augmentation(inputs) if apply_augmentation else inputs
    x = densenet_preprocess(x)
    base_model = DenseNet121(weights="imagenet", include_top=False, input_tensor=x)
    base_model.trainable = False
    x = layers.GlobalAveragePooling2D(name="gap")(base_model.output)
    x = layers.Dense(256, activation="relu", name="dense_256")(x)
    x = layers.Dropout(0.5, name="dropout_0.5")(x)
    outputs = layers.Dense(len(TARGET_CLASSES), activation="softmax", name="predictions")(x)
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name=name)
    return model, base_model

def build_mobilenet_model(name="MobileNet-V2", apply_augmentation=True):
    inputs = layers.Input(shape=(224, 224, 3))
    x = data_augmentation(inputs) if apply_augmentation else inputs
    x = mobilenet_preprocess(x)
    base_model = MobileNetV2(weights="imagenet", include_top=False, input_tensor=x)
    base_model.trainable = False
    x = layers.GlobalAveragePooling2D(name="gap")(base_model.output)
    x = layers.Dense(256, activation="relu", name="dense_256")(x)
    x = layers.Dropout(0.5, name="dropout_0.5")(x)
    outputs = layers.Dense(len(TARGET_CLASSES), activation="softmax", name="predictions")(x)
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name=name)


# 5. Baseline DenseNet-121 Training
**Purpose:** Standard transfer learning, no optimization. CCE loss, 50 epochs, frozen backbone.

In [ ]:
# SUMMARY: Baseline DenseNet-121 Training

baseline_densenet_model, baseline_densenet_base = build_densenet_model(name="Baseline_DenseNet-121")
baseline_densenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=["accuracy"]
)

checkpoint_path = os.path.join(CHECKPOINT_DIR, "Baseline_DenseNet-121_best.keras")
start_epoch = 0
if os.path.exists(checkpoint_path):
    print("Resuming Baseline DenseNet-121 from checkpoint...")
    baseline_densenet_model.load_weights(checkpoint_path)
    start_epoch = get_last_completed_epoch("Baseline_DenseNet-121")

baseline_densenet_history = baseline_densenet_model.fit(
    train_ds, validation_data=val_ds, epochs=50,
    initial_epoch=start_epoch,
    callbacks=get_training_callbacks("Baseline_DenseNet-121")
)


# 6. Optimized DenseNet-121 Training
**Purpose:** Focal Loss + 3-Stage Progressive Unfreezing. Total 100 epochs.

In [ ]:
# SUMMARY: Optimized DenseNet-121 Training (Step A and Step B)

# --- Step A: 50 epochs, frozen backbone, Focal Loss ---
opt_densenet_model, opt_densenet_base = build_densenet_model(name="Optimized_DenseNet-121")
opt_densenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=CategoricalFocalLoss(gamma=2.0, alpha=0.25),
    metrics=["accuracy"]
)

checkpoint_path_A = os.path.join(CHECKPOINT_DIR, "Optimized_DenseNet-121_StepA_best.keras")
start_epoch_A = 0
if os.path.exists(checkpoint_path_A):
    print("Resuming Optimized DenseNet-121 Step A from checkpoint...")
    opt_densenet_model.load_weights(checkpoint_path_A)
    start_epoch_A = get_last_completed_epoch("Optimized_DenseNet-121_StepA")

opt_densenet_history_A = opt_densenet_model.fit(
    train_ds, validation_data=val_ds, epochs=50,
    initial_epoch=start_epoch_A,
    callbacks=get_training_callbacks("Optimized_DenseNet-121_StepA")
)

# --- Step B: Progressive Unfreezing (+50 epochs) ---
# Create shared callbacks so 'best' is tracked continuously across all 3 stages
step_b_callbacks = get_training_callbacks("Optimized_DenseNet-121_StepB")

# Stage 1: epochs 1-17 (only head trainable)
opt_densenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=CategoricalFocalLoss(gamma=2.0, alpha=0.25),
    metrics=["accuracy"]
)
checkpoint_path_B = os.path.join(CHECKPOINT_DIR, "Optimized_DenseNet-121_StepB_best.keras")
start_epoch_B1 = 0
if os.path.exists(checkpoint_path_B):
    print("Resuming Optimized DenseNet-121 Step B Stage 1 from checkpoint...")
    opt_densenet_model.load_weights(checkpoint_path_B)
    start_epoch_B1 = get_last_completed_epoch("Optimized_DenseNet-121_StepB")

opt_densenet_history_B1 = opt_densenet_model.fit(
    train_ds, validation_data=val_ds, epochs=17,
    initial_epoch=start_epoch_B1,
    callbacks=step_b_callbacks
)

# Stage 2: epochs 18-34 (unfreeze Dense Block 4 + Transition 3)
opt_densenet_base.trainable = True
for layer in opt_densenet_base.layers:
    if not layer.name.startswith(('conv5', 'pool4')):
        layer.trainable = False
opt_densenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=CategoricalFocalLoss(gamma=2.0, alpha=0.25),
    metrics=["accuracy"]
)
start_epoch_B2 = max(17, get_last_completed_epoch("Optimized_DenseNet-121_StepB"))
if os.path.exists(checkpoint_path_B) and start_epoch_B2 > 17:
    print("Resuming Optimized DenseNet-121 Step B Stage 2 from checkpoint...")
    opt_densenet_model.load_weights(checkpoint_path_B)

opt_densenet_history_B2 = opt_densenet_model.fit(
    train_ds, validation_data=val_ds, epochs=34,
    initial_epoch=start_epoch_B2,
    callbacks=step_b_callbacks
)

# Stage 3: epochs 35-50 (unfreeze Dense Block 3 + Transition 2)
for layer in opt_densenet_base.layers:
    if layer.name.startswith(('conv4', 'pool3', 'conv5', 'pool4')):
        layer.trainable = True
opt_densenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=CategoricalFocalLoss(gamma=2.0, alpha=0.25),
    metrics=["accuracy"]
)
start_epoch_B3 = max(34, get_last_completed_epoch("Optimized_DenseNet-121_StepB"))
if os.path.exists(checkpoint_path_B) and start_epoch_B3 > 34:
    print("Resuming Optimized DenseNet-121 Step B Stage 3 from checkpoint...")
    opt_densenet_model.load_weights(checkpoint_path_B)

opt_densenet_history_B3 = opt_densenet_model.fit(
    train_ds, validation_data=val_ds, epochs=50,
    initial_epoch=start_epoch_B3,
    callbacks=step_b_callbacks
)


# 7. Baseline MobileNet-V2 Training
**Purpose:** MobileNet-V2 lightweight baseline, standard transfer learning. CCE loss, 50 epochs, frozen backbone.

In [ ]:
# SUMMARY: Baseline MobileNet-V2 Training

baseline_mobilenet_model, baseline_mobilenet_base = build_mobilenet_model(name="Baseline_MobileNet-V2")
baseline_mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=["accuracy"]
)

checkpoint_path = os.path.join(CHECKPOINT_DIR, "Baseline_MobileNet-V2_best.keras")
start_epoch = 0
if os.path.exists(checkpoint_path):
    print("Resuming Baseline MobileNet-V2 from checkpoint...")
    baseline_mobilenet_model.load_weights(checkpoint_path)
    start_epoch = get_last_completed_epoch("Baseline_MobileNet-V2")

baseline_mobilenet_history = baseline_mobilenet_model.fit(
    train_ds, validation_data=val_ds, epochs=50,
    initial_epoch=start_epoch,
    callbacks=get_training_callbacks("Baseline_MobileNet-V2")
)


# Install Required Libraries
**Purpose:** Installs dependencies for evaluation and statistical testing.

In [ ]:
!pip install scikit-posthocs keras-flops

# 8. Shared Evaluation Harness
**Purpose:** Compute Accuracy, Macro P/R/F1, AUC-ROC (OvR, macro), Inference Time, Model Size, FLOPs, and Confusion Matrices.

In [ ]:
import numpy as np
import pandas as pd
import time
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, accuracy_score

try:
    from keras_flops import get_flops
except ImportError:
    get_flops = None

# Helper to calculate flops if library unavailable
def calculate_flops_fallback(model):
    if get_flops:
        try:
            return get_flops(model, batch_size=1)
        except Exception as e:
            print(f"Warning: get_flops failed ({e}). FLOPs will be reported as 0.")
    else:
        print("Warning: keras_flops library is unavailable. FLOPs will be reported as 0.")
    return 0
def evaluate_model(model, name, test_ds, checkpoint_path):
    # 1. Load best weights
    if os.path.exists(checkpoint_path):
        model.load_weights(checkpoint_path)
    
    # 2. Predictions and truth
    test_labels = np.concatenate([y for x, y in test_ds], axis=0)
    true_classes = np.argmax(test_labels, axis=1)
    
    predictions = model.predict(test_ds, verbose=0)
    predicted_classes = np.argmax(predictions, axis=1)
    
    # 3. Metrics
    acc = accuracy_score(true_classes, predicted_classes)
    report = classification_report(true_classes, predicted_classes, target_names=TARGET_CLASSES, output_dict=True)
    macro_p = report['macro avg']['precision']
    macro_r = report['macro avg']['recall']
    macro_f1 = report['macro avg']['f1-score']
    
    auc_roc = roc_auc_score(test_labels, predictions, multi_class='ovr', average='macro')
    
    # 4. Inference Time
    sample_image = next(iter(test_ds))[0][0:1]
    # Warmup
    for _ in range(10):
        model(sample_image, training=False)
    # Timed passes
    start_time = time.time()
    for _ in range(100):
        model(sample_image, training=False)
    end_time = time.time()
    mean_inference_time_ms = ((end_time - start_time) / 100) * 1000
    
    # 5. Model Size
    model_size_mb = os.path.getsize(checkpoint_path) / (1024 * 1024) if os.path.exists(checkpoint_path) else 0
    
    # 6. FLOPs
    flops = calculate_flops_fallback(model)
    
    # 7. Confusion Matrix Plot
    cm = confusion_matrix(true_classes, predicted_classes)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=TARGET_CLASSES, yticklabels=TARGET_CLASSES)
    plt.title(f'Confusion Matrix: {name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'cm_{name}.png')
    plt.show()
    
    return {
        "Model": name,
        "Accuracy": acc,
        "Macro Precision": macro_p,
        "Macro Recall": macro_r,
        "Macro F1": macro_f1,
        "AUC-ROC (macro)": auc_roc,
        "Inference Time (ms)": mean_inference_time_ms,
        "Model Size (MB)": model_size_mb,
        "FLOPs": flops
    }

results = []
results.append(evaluate_model(baseline_densenet_model, "Baseline DenseNet-121", test_ds, os.path.join(CHECKPOINT_DIR, "Baseline_DenseNet-121_best.keras")))
results.append(evaluate_model(opt_densenet_model, "Optimized DenseNet-121", test_ds, os.path.join(CHECKPOINT_DIR, "Optimized_DenseNet-121_StepB_best.keras")))
results.append(evaluate_model(baseline_mobilenet_model, "Baseline MobileNet-V2", test_ds, os.path.join(CHECKPOINT_DIR, "Baseline_MobileNet-V2_best.keras")))

df_results = pd.DataFrame(results)
display(df_results)


# 9. Statistical Testing
**Purpose:** Shapiro-Wilk for normality, followed by ANOVA or Kruskal-Wallis, then post-hoc tests.

In [ ]:
from scipy.stats import shapiro, f_oneway, kruskal
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import scikit_posthocs as sp
import numpy as np
from sklearn.metrics import classification_report

# Perform paired bootstrap resampling: same resampled indices are used across all three models
# in each iteration to ensure a direct apples-to-apples comparison on the exact same subsets.
def paired_bootstrap_f1(models_dict, test_ds, n_iterations=30):
    test_labels = np.concatenate([y for x, y in test_ds], axis=0)
    true_classes = np.argmax(test_labels, axis=1)
    
    # Pre-compute all predictions to save time
    preds = {}
    for name, model in models_dict.items():
        predictions = model.predict(test_ds, verbose=0)
        preds[name] = np.argmax(predictions, axis=1)
    
    f1_scores = {name: [] for name in models_dict.keys()}
    np.random.seed(42)
    
    for _ in range(n_iterations):
        # Shared random index draw for this iteration
        indices = np.random.choice(len(true_classes), len(true_classes), replace=True)
        sample_true = true_classes[indices]
        
        # Compute metrics for all models on the identical resampled indices
        for name in models_dict.keys():
            sample_pred = preds[name][indices]
            rep = classification_report(sample_true, sample_pred, output_dict=True, zero_division=0)
            f1_scores[name].append(rep['macro avg']['f1-score'])
            
    return f1_scores

models = {
    'Base_Dense': baseline_densenet_model,
    'Opt_Dense': opt_densenet_model,
    'Base_Mobile': baseline_mobilenet_model
}

f1_results = paired_bootstrap_f1(models, test_ds)
f1_base_dense = np.array(f1_results['Base_Dense'])
f1_opt_dense = np.array(f1_results['Opt_Dense'])
f1_base_mobile = np.array(f1_results['Base_Mobile'])

print("--- Shapiro-Wilk Normality Test ---")
stat1, p1 = shapiro(f1_base_dense)
stat2, p2 = shapiro(f1_opt_dense)
stat3, p3 = shapiro(f1_base_mobile)
print(f"Baseline DenseNet-121: W={stat1:.4f}, p={p1:.4f}")
print(f"Optimized DenseNet-121: W={stat2:.4f}, p={p2:.4f}")
print(f"Baseline MobileNet-V2: W={stat3:.4f}, p={p3:.4f}")

is_normal = (p1 > 0.05) and (p2 > 0.05) and (p3 > 0.05)

print("\n--- Omnibus Test ---")
if is_normal:
    print("Distributions are normal. Proceeding with one-way ANOVA.")
    stat, p_val = f_oneway(f1_base_dense, f1_opt_dense, f1_base_mobile)
    print(f"ANOVA F-stat={stat:.4f}, p-value={p_val:.4e}")
    if p_val < 0.05:
        print("\n--- Tukey HSD Post-Hoc ---")
        data = np.concatenate([f1_base_dense, f1_opt_dense, f1_base_mobile])
        labels = ['Base_Dense']*30 + ['Opt_Dense']*30 + ['Base_Mobile']*30
        tukey = pairwise_tukeyhsd(endog=data, groups=labels, alpha=0.05)
        print(tukey)
else:
    print("Distributions are not all normal. Proceeding with Kruskal-Wallis.")
    stat, p_val = kruskal(f1_base_dense, f1_opt_dense, f1_base_mobile)
    print(f"Kruskal-Wallis H-stat={stat:.4f}, p-value={p_val:.4e}")
    if p_val < 0.05:
        print("\n--- Dunn's Post-Hoc (Bonferroni) ---")
        data = [f1_base_dense, f1_opt_dense, f1_base_mobile]
        p_values = sp.posthoc_dunn(data, p_adjust='bonferroni')
        print("Groups: 1=Base_Dense, 2=Opt_Dense, 3=Base_Mobile")
        print(p_values)
